In [16]:
#Bài 3
import copy
import math
import numpy as np

X = "X"
O = "O"
EMPTY = None

class TicTacToe:
    def __init__(self, n=5, win_length=4):
        self.n = n
        self.win_length = win_length
        self.board = [[EMPTY for _ in range(n)] for _ in range(n)]

    def player(self, board):
        count = sum(1 for row in board for cell in row if cell is not None)
        return O if count % 2 != 0 else X

    def actions(self, board):
        # Tối ưu: Chỉ tìm các ô trống
        return [(r, c) for r in range(self.n) for c in range(self.n) if board[r][c] == EMPTY]

    def result(self, board, action):
        new_board = [row[:] for row in board]
        new_board[action[0]][action[1]] = self.player(board)
        return new_board

    def winner(self, board):
        # Kiểm tra hàng và cột
        for i in range(self.n):
            for j in range(self.n - self.win_length + 1):
                # Hàng
                window_h = board[i][j:j + self.win_length]
                if window_h.count(X) == self.win_length: return X
                if window_h.count(O) == self.win_length: return O
                # Cột
                window_v = [board[k][i] for k in range(j, j + self.win_length)]
                if window_v.count(X) == self.win_length: return X
                if window_v.count(O) == self.win_length: return O

        # Kiểm tra đường chéo
        for i in range(self.n - self.win_length + 1):
            for j in range(self.n - self.win_length + 1):
                diag1 = [board[i+k][j+k] for k in range(self.win_length)]
                diag2 = [board[i+k][j+self.win_length-1-k] for k in range(self.win_length)]
                if diag1.count(X) == self.win_length or diag2.count(X) == self.win_length: return X
                if diag1.count(O) == self.win_length or diag2.count(O) == self.win_length: return O
        return None

    def terminal(self, board):
        if self.winner(board): return True
        return all(cell is not EMPTY for row in board for cell in row)

    def utility(self, board):
        res = self.winner(board)
        if res == X: return 100
        if res == O: return -100
        return 0

    def minimax(self, board, depth, alpha, beta, is_maximizing):
        """Thuật toán Minimax kết hợp Alpha-Beta và Depth Limit"""
        if self.terminal(board) or depth == 0:
            return self.utility(board)

        if is_maximizing:
            v = -math.inf
            for action in self.actions(board):
                v = max(v, self.minimax(self.result(board, action), depth - 1, alpha, beta, False))
                alpha = max(alpha, v)
                if beta <= alpha: break
            return v
        else:
            v = math.inf
            for action in self.actions(board):
                v = min(v, self.minimax(self.result(board, action), depth - 1, alpha, beta, True))
                beta = min(beta, v)
                if beta <= alpha: break
            return v

    def get_best_move(self, board):
        best_v = -math.inf
        best_move = None
        # Độ sâu khuyến nghị: NxN càng lớn depth càng nhỏ để tránh lag
        depth = 4 if self.n <= 5 else 3

        for action in self.actions(board):
            v = self.minimax(self.result(board, action), depth, -math.inf, math.inf, False)
            if v > best_v:
                best_v = v
                best_move = action
        return best_move

# --- Chương trình chính ---
if __name__ == "__main__":
    N = int(input("Nhập kích thước bàn cờ N: "))
    WIN = int(input("Nhập số quân liên tiếp để thắng: "))
    game = TicTacToe(n=N, win_length=WIN)
    current_board = game.board

    print(f"Bàn cờ {N}x{N}, thắng khi đủ {WIN} quân.")

    while not game.terminal(current_board):
        print(np.array(current_board))
        p = game.player(current_board)

        if p == X: # AI lượt X
            print("AI đang suy nghĩ...")
            move = game.get_best_move(current_board)
            current_board = game.result(current_board, move)
        else: # Người lượt O
            r = int(input("Dòng: "))
            c = int(input("Cột: "))
            if current_board[r][c] == EMPTY:
                current_board = game.result(current_board, (r, c))
            else:
                print("Ô đã có quân!")
                continue

    print(np.array(current_board))
    winner = game.winner(current_board)
    print(f"Kết quả: {winner if winner else 'Hòa'}")

Nhập kích thước bàn cờ N: 3
Nhập số quân liên tiếp để thắng: 3
Bàn cờ 3x3, thắng khi đủ 3 quân.
[[None None None]
 [None None None]
 [None None None]]
AI đang suy nghĩ...
[['X' None None]
 [None None None]
 [None None None]]
Dòng: 1
Cột: 1
[['X' None None]
 [None 'O' None]
 [None None None]]
AI đang suy nghĩ...
[['X' 'X' None]
 [None 'O' None]
 [None None None]]
Dòng: 0
Cột: 2
[['X' 'X' 'O']
 [None 'O' None]
 [None None None]]
AI đang suy nghĩ...
[['X' 'X' 'O']
 [None 'O' None]
 ['X' None None]]
Dòng: 1
Cột: 0
[['X' 'X' 'O']
 ['O' 'O' None]
 ['X' None None]]
AI đang suy nghĩ...
[['X' 'X' 'O']
 ['O' 'O' 'X']
 ['X' None None]]
Dòng: 2
Cột: 1
[['X' 'X' 'O']
 ['O' 'O' 'X']
 ['X' 'O' None]]
AI đang suy nghĩ...
[['X' 'X' 'O']
 ['O' 'O' 'X']
 ['X' 'O' 'X']]
Kết quả: Hòa
